# Problem Statement: Linear Programming Optimization

## Objective
**Maximize** the objective function:
$$Z = x_1 + 3x_2$$

## Constraints
Subject to the following linear inequalities:
1.  $-x_1 + x_2 \leq 3$
2.  $-x_1 + 2x_2 \leq 8$
3.  $3x_1 + x_2 \leq 18$
4.  $x_1, x_2 \geq 0$


## 1. Simplex Method in matrix form

### 1.1 First Pass:

### Solution: Simplex Iteration: Basic Feasible Solution (BFS)

**Basis Selection:** The basis is defined as $B = (x_2, x_4, x_5)$, where:
* $x_2$: Decision variable 2
* $x_4$: Slack variable for Constraint 2
* $x_5$: Slack variable for Constraint 3

In [8]:
import numpy as np

In [9]:
# Basis Matrix (B)
a_b = np.array([[1,0,0],
                [2,1,0],
                [1,0,1]])


In [10]:
# Non-basis Matrix (N)
a_n = np.array([[-1,1],
                [-1,0],
                [3,0]])

In [11]:
# RHS vector and partitioned objective function coefficients (costs)
b = np.array([3,8,18]).T
c_b = np.array([3,0,0]).T
c_n = np.array([1,0]).T

In [12]:
# Calculate values of basic variables for the current Basic Feasible Solution (BFS)
x_b = np.linalg.inv(a_b) @ b
print(x_b)

[ 3.  2. 15.]


In [13]:
# Reduced costs
'''
Calculate reduced costs for non-basic variables (z_j - c_j)
A negative value (like -4.0) indicates a candidate to enter the basis for maximization.
'''
cn_T_bar = c_b.T @ np.linalg.inv(a_b) @ a_n - c_n.T
print(cn_T_bar)

[-4.  3.]


In [14]:
# Pivot column
# Calculate the entering column in terms of the current basis (B_inv * a_j)
# This determines the direction of movement toward the next vertex.
pivot = np.linalg.inv(a_b) @ a_n[:,0] # select the first column
print(pivot)

[-1.  1.  4.]


The ratio test:
Divide x_b vales (bfs) by these positive pivot values. The one basis variable corresponding to the minimum ratio leaves.
2/1 < 15/4, so x4 leaves.

### 1.2 Second Pass

From the first pass, x4 should be leave, and x1 should enter. Now the basic variables are (x1, x2, x5)

In [15]:
# Basis Matrix (B)
a_b = np.array([[-1,1,0],
                [-1,2,0],
                [3,1,1]])

In [16]:
# Non-basis Matrix (N)
a_n = np.array([[1,0],
                [0,1],
                [0,0]])

In [17]:
b = np.array([3,8,18]).T
c_b = np.array([1,3,0]).T
c_n = np.array([0,0]).T

In [18]:
x_b = np.linalg.inv(a_b) @ b
print(x_b)

[2. 5. 7.]


In [19]:
cn_T_bar = c_b.T @ np.linalg.inv(a_b) @ a_n - c_n.T
print(cn_T_bar)

[-5.  4.]


In [20]:
pivot = np.linalg.inv(a_b) @ a_n[:,0] # select the first column
print(pivot)

[-2. -1.  7.]


Only one positive pivot value, so x5 leaves.

### 1.3 Third Pass

From the first pass, x5 should be removed, and x3 should enter. Now the basic variables are (x1, x2, x3)

In [21]:
a_b = np.array([[-1,1,1],
                [-1,2,0],
                [3,1,0]])

In [22]:
a_n = np.array([[0,0],
                [1,0],
                [0,1]])

In [23]:
b = np.array([3,8,18]).T
c_b = np.array([1,3,0]).T
c_n = np.array([0,0]).T

In [24]:
x_b = np.linalg.inv(a_b) @ b
print(x_b)

[4. 6. 1.]


In [25]:
cn_T_bar = c_b.T @ np.linalg.inv(a_b) @ a_n - c_n.T
print(cn_T_bar)

[1.14285714 0.71428571]


The reduced costs are all positive, and we get the optimal solution (4, 6, 1) for the basic variables, and zeros for the non-basic variables.

The optimal solutioon is x1=4, x2=6

## 2. Solution by Gurobypy

In [26]:
import gurobipy as gp
from gurobipy import GRB

# 1. Create a new model
model = gp.Model("Linear_Program_Example")

# 2. Create variables
# x1 and x2 are non-negative by default (lb=0.0)
x1 = model.addVar(name="x1")
x2 = model.addVar(name="x2")

# 3. Set objective: Maximize x1 + 3*x2
model.setObjective(x1 + 3*x2, GRB.MAXIMIZE)

# 4. Add constraints
model.addConstr(-x1 + x2 <= 3, "c1")
model.addConstr(-x1 + 2*x2 <= 8, "c2")
model.addConstr(3*x1 + x2 <= 18, "c3")

# 5. Optimize the model
model.optimize()

# 6. Print the results
if model.Status == GRB.OPTIMAL:
    print(f"\nOptimal Objective Value (Z): {model.ObjVal}")
    print(f"Optimal x1: {x1.X}")
    print(f"Optimal x2: {x2.X}")

    # Optional: Print slack variables
    for c in model.getConstrs():
        print(f"Slack for {c.ConstrName}: {c.Slack}")

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5950X 16-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 32 logical processors, using up to 32 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model fingerprint: 0xd1e73bf2
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 2e+01]
Presolve removed 3 rows and 2 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.2000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.200000000e+01

Optimal Objective Value (Z): 22.0
Optimal x1: 4.0
Optimal x2: 6.0
Slack for c1: 1.0
Slack for c2: 0.0
Slack for c3: 0.0
